# Local CPU preparation — WSL offline

Mở `full-pipeline.ipynb` làm notebook đầu tiên. Bước 2 được chạy ở notebook này trong **WSL local**, dùng staging workspace dưới `/home` để giảm I/O trên `/mnt/d`. Không cần Colab, Google Drive, Kaggle, pip/apt download hay internet cho bước CPU. Cache/input lấy từ bundle local; checkpoint và prepared output được ghi về `D:/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/bundles/`.

Chọn kernel WSL có sẵn `numpy`, `PyYAML`, `rapidfuzz`, `python-dotenv`, `tqdm`; Java 21 và Joern runtime cần có sẵn hoặc nằm trong runtime zip local. Notebook không tự cài gói từ mạng.


In [3]:
from pathlib import Path
import os,sys,shutil,subprocess,json,zipfile,hashlib

if os.name == 'nt':
    raise RuntimeError('Mở notebook bằng kernel WSL/Linux, không phải Windows kernel.')
PROJECT_ROOT = Path(os.environ.get('GRACE_H1_ROOT', '/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection')).resolve()
if not (PROJECT_ROOT/'grace_h1').is_dir():
    raise FileNotFoundError(f'Project không tồn tại: {PROJECT_ROOT}')
WORK_ROOT = Path.home()/'GRACE-H1-SecurityProjection-reveal-test'
BUNDLES = PROJECT_ROOT/'bundles'
INPUT_ZIP = BUNDLES/'projection-v2-cpu-inputs.zip'
HEADS_ZIP = BUNDLES/'projection-v2-heads.zip'
JOERN_ZIP = BUNDLES/'grace-h1-joern-runtime.zip'
PREPARED_OUT = BUNDLES/'projection-v2-cpu-prepared.zip'
CHECKPOINT_OUT = BUNDLES/'projection-v2-cpu-checkpoint.zip'
print('Project/export:', PROJECT_ROOT)
print('WSL work root:', WORK_ROOT)
print('Free space near /home:', f'{shutil.disk_usage(Path.home()).free/1024**3:.1f} GiB')
for p in (INPUT_ZIP, HEADS_ZIP, JOERN_ZIP):
    print(p.name, p.exists(), f'{p.stat().st_size:,} bytes' if p.exists() else '')
if any(not p.is_file() for p in (INPUT_ZIP,HEADS_ZIP,JOERN_ZIP)):
    raise FileNotFoundError('Cần các input bundle local trong bundles/.')
if shutil.disk_usage(Path.home()).free < 3*1024**3:
    raise RuntimeError('Cần tối thiểu 3 GiB trống dưới /home để giải nén cache và chạy Joern.')


Project/export: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection
WSL work root: /home/yungn/GRACE-H1-SecurityProjection-reveal-test
Free space near /home: 861.7 GiB
projection-v2-cpu-inputs.zip True 506,201,004 bytes
projection-v2-heads.zip True 23,200,995 bytes
grace-h1-joern-runtime.zip True 238,055,408 bytes


In [4]:
# Lấy đúng các module Python từ bundle local; không cài hoặc tải gì qua mạng.
def bootstrap_sources(root, archive_path):
    root.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        manifest=json.loads(archive.read('bundle_manifest.json'))
        if manifest.get('schema')!='grace_projection_v2_bundle_v1' or manifest.get('kind')!='cpu_inputs':
            raise ValueError('Bundle không phải Projection V2 CPU input')
        for entry in manifest['entries']:
            name=entry['path']; parts=Path(name).parts
            source_file=name.startswith('grace_h1/') and name.endswith('.py')
            if not source_file: continue
            if Path(name).is_absolute() or '..' in parts or ':' in name or '\\' in name:
                raise ValueError(f'Unsafe source path: {name}')
            data=archive.read(name)
            if len(data)!=entry['bytes'] or hashlib.sha256(data).hexdigest()!=entry['sha256']:
                raise ValueError(f'Source checksum mismatch: {name}')
            target=root/name;target.parent.mkdir(parents=True,exist_ok=True)
            if target.exists() and target.read_bytes()!=data:
                raise ValueError(f'Source khác trong {root}; giữ folder cũ và đặt GRACE_H1_ROOT mới.')
            target.write_bytes(data)
    if str(root) not in sys.path: sys.path.insert(0,str(root))

bootstrap_sources(WORK_ROOT,INPUT_ZIP)
import numpy, yaml, rapidfuzz, dotenv, tqdm
print('Dependencies đã có trong kernel; không gọi pip/apt.')


Dependencies đã có trong kernel; không gọi pip/apt.


In [5]:
from grace_h1.portable_v2 import import_bundle,create_bundle
from grace_h1.portable import import_bundle as import_joern_bundle
from grace_h1.experiment_v2 import load_config,validate_config,require_heads,prepare_all
from grace_h1.io import load_local_environment

load_local_environment(WORK_ROOT)
print('CPU inputs:',import_bundle(WORK_ROOT,INPUT_ZIP,'cpu_inputs'))
print('Trained heads:',import_bundle(WORK_ROOT,HEADS_ZIP,'heads'))
print('Joern runtime:',import_joern_bundle(WORK_ROOT,JOERN_ZIP,'joern_runtime'))

JOERN_HOME=WORK_ROOT/'tools/joern/joern-cli'
if not (JOERN_HOME/'joern-parse').is_file():
    raise FileNotFoundError(f'Joern runtime thiếu: {JOERN_HOME}')
os.environ['JOERN_HOME']=str(JOERN_HOME)
java=subprocess.run(['java','-version'],capture_output=True,text=True)
java_text=java.stderr+'\n'+java.stdout
print(java_text)
if java.returncode or '21' not in java_text:
    raise RuntimeError('Cần Java 21 đã cài local trong WSL; notebook không cài Java từ internet.')
config=load_config(WORK_ROOT);validate_config(config)
if config['splits']!=['test'] or not config.get('test_protocol_frozen'):
    raise ValueError('CPU bundle chưa khóa ReVeal test.')
selection=require_heads(WORK_ROOT,config)
print('Frozen config:',config['datasets'],config['splits'])
print('Heads:',[(d,a,h['selection_status']) for d,arms in selection['heads'].items() for a,h in arms.items()])


CPU inputs: {'kind': 'cpu_inputs', 'copied': 37188, 'reused_identical': 17, 'updated_selection_pointers': 0, 'archive_sha256': '0ef2a5b4a83a851937a6b774e388f09fd754b1914ae084c11cd34a04796f32de'}
Trained heads: {'kind': 'heads', 'copied': 23, 'reused_identical': 0, 'updated_selection_pointers': 0, 'archive_sha256': '4497acf164548be18ab022253caa257569abb77e9457b2c91f0767e178f880dd'}
Joern runtime: {'kind': 'joern_runtime', 'copied': 219, 'reused_identical': 0, 'archive_sha256': '13637b0450c970b1f0a5b41acae5fbfaad2982956821158ed40161b157b1e6e7'}
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


Frozen config: ['reveal'] ['test']
Heads: [('reveal', 'residual_ce', 'validation_proxy_gain'), ('reveal', 'residual_local', 'validation_proxy_gain')]


## Chạy Bước 2 — cache hit, graph còn thiếu mới parse

Graph cache và embeddings đã được đưa vào `WORK_ROOT` từ bundle local. Progress bar hiển thị cache hit, graph vừa parse và record ID đang chạy. Tác vụ checkpoint ghi bundle vào thư mục local của dự án; có thể chạy lại notebook trên cùng `WORK_ROOT` sau khi ngắt, graph đã ghi sẽ được nhận lại.


In [6]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from tqdm.auto import tqdm
from grace_h1 import pipeline as pipeline_module

class ProgressPool(ThreadPoolExecutor):
    def map(self,fn,rows,timeout=None,chunksize=1):
        lock=Lock();state={'cache':0,'parsed':0};active={}
        bar=tqdm(total=len(rows),desc='Graph preparation',unit='graph',dynamic_ncols=True)
        def tracked(row):
            h=row['code_sha256'];folder=WORK_ROOT/'cache/graphs'/h
            cached=(folder/'features.json.gz').exists() or (folder/'features.json').exists()
            with lock:
                active[h]=row['record_id']
                bar.set_postfix_str(f"cache {state['cache']} | parsed {state['parsed']} | đang chạy {len(active)}: {str(list(active.values())[:2])[:100]}")
            try:
                result=fn(row)
                with lock:
                    state['cache' if cached else 'parsed']+=1
                    bar.update(1)
                return result
            finally:
                with lock:
                    active.pop(h,None)
                    bar.set_postfix_str(f"cache {state['cache']} | parsed {state['parsed']} | đang chạy {len(active)}: {str(list(active.values())[:2])[:100]}")
        def results():
            try: yield from super(ProgressPool,self).map(tracked,rows,timeout=timeout,chunksize=chunksize)
            finally: bar.close()
        return results()

pipeline_module.ThreadPoolExecutor=ProgressPool

def checkpoint_local():
    local=WORK_ROOT/'bundles/projection-v2-cpu-checkpoint.zip'
    info=create_bundle(WORK_ROOT,local,'checkpoint')
    temporary=CHECKPOINT_OUT.with_suffix('.tmp.zip')
    shutil.copyfile(local,temporary);os.replace(temporary,CHECKPOINT_OUT)
    tqdm.write(f"Checkpoint local: {CHECKPOINT_OUT} ({info['bytes']:,} bytes)")

try:
    run_rows=prepare_all(WORK_ROOT,config,checkpoint_callback=checkpoint_local)
finally:
    checkpoint_local()
print(run_rows)


/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


reveal/baseline_whitening: 9819 unique graph features required


Graph preparation:   5%|▌         | 501/9819 [00:04<01:34, 99.08graph/s, cache 501 | parsed 0 | đang chạy 1: ['reveal:test:-3740862514502467070#500']]                                          

graphs 500/9819


Graph preparation: 100%|██████████| 9819/9819 [02:03<00:00, 93.33graph/s, cache 9819 | parsed 0 | đang chạy 0: []]                                                                                     

...

graphs 9500/9754


Graph preparation:  99%|█████████▉| 9663/9754 [11:26<00:13,  6.89graph/s, cache 9484 | parsed 179 | đang chạy 3: ['reveal:train:-281311800725760049#2047', 'reveal:train:174091092490434632#6479']]   

Checkpoint local: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/bundles/projection-v2-cpu-checkpoint.zip (67,504,330 bytes)


Graph preparation: 100%|██████████| 9754/9754 [12:01<00:00, 13.52graph/s, cache 9566 | parsed 188 | đang chạy 0: []]                                                                                   


prepared 2274 prompts; graph coverage 95.1%: /home/yungn/GRACE-H1-SecurityProjection-reveal-test/runs/reveal/test/h1_residual_local/98d7f32cd098d4c0
Checkpoint local: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/bundles/projection-v2-cpu-checkpoint.zip (67,646,900 bytes)
Checkpoint local: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/bundles/projection-v2-cpu-checkpoint.zip (73,461,828 bytes)
[{'dataset': 'reveal', 'split': 'test', 'arm': 'baseline_whitening', 'path': 'runs/reveal/test/baseline_whitening/2d1845eabf62eb84'}, {'dataset': 'reveal', 'split': 'test', 'arm': 'h1_residual_local', 'path': 'runs/reveal/test/h1_residual_local/98d7f32cd098d4c0'}]


## Xuất kết quả về thư mục local của dự án

Prepared bundle được tạo trong WSL work root, sau đó copy nguyên file về `bundles/projection-v2-cpu-prepared.zip` trên ổ D. Không dùng Google Drive. Sau khi cell này thành công, quay lại `full-pipeline.ipynb` và chạy lại từ đầu.


In [7]:
local_prepared=WORK_ROOT/'bundles/projection-v2-cpu-prepared.zip'
info=create_bundle(WORK_ROOT,local_prepared,'prepared')
temporary=PREPARED_OUT.with_suffix('.tmp.zip')
shutil.copyfile(local_prepared,temporary);os.replace(temporary,PREPARED_OUT)
print({'local_path':str(PREPARED_OUT),'bytes':PREPARED_OUT.stat().st_size,'sha256':info['sha256']})


{'local_path': '/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/bundles/projection-v2-cpu-prepared.zip', 'bytes': 73461822, 'sha256': 'f6b7ec7f495d741f2bfbaeae429cf5c56f9c64fb4866aa2b28753caccbbbb95a'}
